# RepCount Part-A — Data Preparation

This notebook is the **data preparation pipeline** separated from EDA.

**Why this notebook exists**
- EDA should diagnose issues; preparation should apply deterministic fixes and export model-ready data.
- Keeping them separate avoids accidental metric drift and improves reproducibility.

**Current decisions in this version**
- Normalize known typo labels.
- Apply manual review decisions from `others` inspection.
- Keep all exercise classes by default (no active exclusions).
- Keep exclusion hooks available for domain/policy experiments.


**Next steps after running this notebook**
- Use exported cleaned CSVs in MediaPipe and YOLOv8 data pipelines.
- Track this policy in experiment metadata for both modeling tracks.
- Re-run when label rules or class scope changes.


## Project Goal and Success Metrics

### Goal
**Real-Time Rep Counter via Human Pose Estimation**

### Core pipeline objective
1. **Detect the person** in each frame (or track the main subject).
2. **Detect movement dynamics** from pose over time.
3. **Count completed repetitions** robustly in real time.

### Primary metric
- **Rep Count MAE (Mean Absolute Error)** per video/session.

### Required supporting metrics
- **Per-class MAE** (to reveal class-specific weakness and imbalance effects).
- **Real-time performance** (latency/FPS for deployment readiness).

### Optional reporting metrics
- **RMSE** on rep count.
- **Within-1 rep accuracy**.



In [ ]:
import os
import pandas as pd
import numpy as np

print('Libraries loaded')


## 1) Paths and Config

**Why this section**
- Centralizes file locations and policy switches so the pipeline is easy to audit and change.

**Decision(s)**
- Input comes from raw LLSP annotation CSVs.
- Output is written to `annotation_cleaned/`.
- Keep exclusion hooks available, but empty by default.

**Next step(s)**
- Update paths if data storage changes.
- Add classes to exclusion lists only when a run explicitly requires it.


In [ ]:
TRAIN_PATH = '../../Data/LLSP/annotation/train.csv'
VALID_PATH = '../../Data/LLSP/annotation/valid.csv'

OUTPUT_DIR = '../../Data/LLSP/annotation_cleaned'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Optional exclusion hooks (kept empty by default)
DOMAIN_EXCLUDE_TYPES = []   # e.g., ['pommelhorse']
POLICY_EXCLUDE_TYPES = []   # e.g., ['battle_rope']
EXCLUDE_TYPES = sorted(set(DOMAIN_EXCLUDE_TYPES + POLICY_EXCLUDE_TYPES))


## 2) Load Splits

**Why this section**
- Establishes a single, explicit source of truth for train/valid before transformation.

**Decision(s)**
- Preserve split identity in each dataframe.
- Keep all raw columns initially (including temporal `L*` fields).
- Remove columns that are entirely null in validation, since they provide no validation signal.

**Next step(s)**
- Verify row counts against expected dataset totals before any cleaning.
- Review dropped-column logs and keep required columns protected.


In [ ]:
def load_split(path, split_name):
    df = pd.read_csv(path)
    if df.columns[0] in ['Unnamed: 0', '']:
        df = df.rename(columns={df.columns[0]: 'index'})
    df['split'] = split_name
    return df


def align_split_schema(train_df, valid_df):
    # Keep union of columns; preserve train column order, then append valid-only columns.
    all_cols = list(train_df.columns) + [c for c in valid_df.columns if c not in train_df.columns]

    for col in all_cols:
        if col not in train_df.columns:
            train_df[col] = pd.NA
        if col not in valid_df.columns:
            valid_df[col] = pd.NA

    train_df = train_df[all_cols].copy()
    valid_df = valid_df[all_cols].copy()
    return train_df, valid_df


def drop_valid_all_null_columns(train_df, valid_df):
    # Protect core fields from accidental drop even if null-heavy.
    protected_cols = {'name', 'type', 'count', 'split', 'index'}
    drop_cols = [
        c for c in valid_df.columns
        if c not in protected_cols and valid_df[c].isna().all()
    ]

    if drop_cols:
        train_df = train_df.drop(columns=drop_cols, errors='ignore')
        valid_df = valid_df.drop(columns=drop_cols, errors='ignore')

    return train_df, valid_df, drop_cols


train_df = load_split(TRAIN_PATH, 'train')
valid_df = load_split(VALID_PATH, 'valid')

train_df, valid_df = align_split_schema(train_df, valid_df)
train_df, valid_df, dropped_valid_null_cols = drop_valid_all_null_columns(train_df, valid_df)

df_all_raw = pd.concat([train_df, valid_df], ignore_index=True)

valid_all_null_cols_retained = [c for c in valid_df.columns if valid_df[c].isna().all()]
train_all_null_cols_retained = [c for c in train_df.columns if train_df[c].isna().all()]

print('Loaded rows:')
print(f"  train={len(train_df)}, valid={len(valid_df)}, total={len(df_all_raw)}")
print(f"Raw unique train labels ({train_df['type'].nunique()}):", sorted(train_df['type'].dropna().unique()))
print(f"Schema columns after drop: {len(train_df.columns)}")
print(f"Dropped columns (all-null in valid): {len(dropped_valid_null_cols)}")
if dropped_valid_null_cols:
    print('  sample dropped columns:', dropped_valid_null_cols[:10])
print(f"All-null columns remaining in valid: {len(valid_all_null_cols_retained)}")
print(f"All-null columns remaining in train: {len(train_all_null_cols_retained)}")
if valid_all_null_cols_retained:
    print('  sample valid all-null retained columns:', valid_all_null_cols_retained[:10])
if train_all_null_cols_retained:
    print('  sample train all-null retained columns:', train_all_null_cols_retained[:10])


## 3) Typo Normalization

**Why this section**
- Label variants fragment classes and corrupt class distributions, sampling, and metrics.

**Decision(s)**
- Apply a fixed typo map (e.g., `squant -> squat`, `situp -> sit_up`, etc.).
- Use the same normalization across train/valid/test for consistency.

**Next step(s)**
- If new variants are found, add them here and re-run full prep + EDA checks.


In [ ]:
TYPO_MAP = {
    'squant'        : 'squat',
    'frontraise'    : 'front_raise',
    'benchpressing' : 'bench_pressing',
    'jumpjacks'     : 'jump_jacks',
    'jump_jack'     : 'jump_jacks',
    'situp'         : 'sit_up',
    'pullsup'       : 'pull_up',
    'pullsups'      : 'pull_up',
    'pullups'       : 'pull_up',
    'pushups'       : 'push_up',
    'push_ups'      : 'push_up',
}

for d in [train_df, valid_df]:
    d['type'] = d['type'].replace(TYPO_MAP)

print('After typo-clean, train labels:', sorted(train_df['type'].dropna().unique()))

# Snapshot class counts right after typo-normalization (before manual/exclusion decisions)
train_counts_after_typo = train_df['type'].value_counts().sort_values(ascending=False).copy()
valid_counts_after_typo = valid_df['type'].value_counts().sort_values(ascending=False).copy()

print('\nBaseline train distribution (after typo-fix, before decisions):')
print(train_counts_after_typo.to_string())



## 4) Manual Review Decisions (`others` + corrections)

**Why this section**
- Human review resolves ambiguous/mislabeled videos that automated rules cannot fix.

**Decision(s)**
- `stu6_11.mp4 -> squat` (manual inspection correction).
- Remove reviewed `others` videos confirmed as ambiguous/noisy/incorrect.
- Keep decision logic explicit via `RELABEL_MAP` and `REMOVE_LIST`.

**Next step(s)**
- Add each new manual decision with evidence (PDF/frame review note).
- Keep low-sample classes unless a separate documented quality issue is found.


In [ ]:
RELABEL_MAP = {
    # confirmed manual correction
    'stu6_11.mp4': 'squat',
    'stu11_10.mp4': 'squat', 'stu11_11.mp4': 'squat', 'stu11_12.mp4': 'squat', 'stu11_13.mp4': 'squat', 'stu11_7.mp4': 'squat', 'stu11_8.mp4': 'squat',
    'stu11_9.mp4': 'squat', 'stu12_0.mp4': 'squat', 'stu12_7.mp4': 'squat', 'stu13_0.mp4': 'squat', 'stu13_4.mp4': 'squat',
}

REMOVE_LIST = [
    # reviewed others removed for quality reasons
    'stu1_28.mp4', 'stu1_29.mp4', 'stu11_0.mp4', 'stu11_1.mp4', 'stu12_2.mp4', 'stu12_6.mp4', 'stu12_8.mp4',
    'stu12_1.mp4', 'stu12_5.mp4', 'stu12_9.mp4', 'stu12_10.mp4', 'stu13_1.mp4', 'stu13_5.mp4', 'stu13_6.mp4',
    'stu11_4.mp4', 'stu11_5.mp4', 'stu11_6.mp4', 'stu12_3.mp4', 'stu12_4.mp4', 'stu13_2.mp4', 'stu13_3.mp4',
    'stu5_28.mp4', 'stu10_32.mp4', 'stu10_33.mp4', 'stu11_2.mp4', 'stu11_3.mp4',


]

def normalize_video_name(x):
    x = str(x).strip()
    return x if x.endswith('.mp4') else f'{x}.mp4'

relabel_norm = {normalize_video_name(k): v for k, v in RELABEL_MAP.items()}
remove_set = {normalize_video_name(x) for x in REMOVE_LIST}

# Apply manual curation ONLY to train/valid
affected_rows = {'train': {'relabel': 0, 'removed': 0}, 'valid': {'relabel': 0, 'removed': 0}}
for d, name in [(train_df, 'train'), (valid_df, 'valid')]:
    # relabel
    relabel_hits = 0
    for video_name, new_label in relabel_norm.items():
        mask = d['name'] == video_name
        if mask.any():
            relabel_hits += int(mask.sum())
            d.loc[mask, 'type'] = new_label

    # remove
    before = len(d)
    d.drop(d[d['name'].isin(remove_set)].index, inplace=True)
    d.reset_index(drop=True, inplace=True)
    removed = before - len(d)

    affected_rows[name]['relabel'] = relabel_hits
    affected_rows[name]['removed'] = removed
    print(f"{name:<5} relabeled rows: {relabel_hits}")
    print(f"{name:<5} removed rows: {removed}")



## 5) Optional Class Exclusion

**Why this section**
- Supports controlled experiments with alternate class scope when needed.

**Decision(s)**
- Keep separate hooks for domain-based and policy-based exclusions.
- Current defaults are empty:
  - `DOMAIN_EXCLUDE_TYPES = []`
  - `POLICY_EXCLUDE_TYPES = []`

**Next step(s)**
- When needed, add classes to one or both lists and rerun the notebook.
- Keep exclusions synchronized with training/evaluation configs and report notes.


In [ ]:
if EXCLUDE_TYPES:
    excluded_rows = {'train': 0, 'valid': 0}
    for d, name in [(train_df, 'train'), (valid_df, 'valid')]:
        before = len(d)
        d.drop(d[d['type'].isin(EXCLUDE_TYPES)].index, inplace=True)
        d.reset_index(drop=True, inplace=True)
        excluded_rows[name] = before - len(d)
        print(f"{name:<5} excluded class rows: {excluded_rows[name]}")
else:
    excluded_rows = {'train': 0, 'valid': 0}
    print('No class exclusions applied')



## Dataset Finalization Decisions

### Why this section exists
This section records **formal data-finalization policy** before modeling so that training artifacts are reproducible and reviewable.

### Final decisions applied in this notebook
1. **Scope policy**
   - This notebook processes **train + valid only**.
   - Test split is intentionally excluded from curation decisions to avoid evaluation leakage.

2. **Label normalization**
   - Known typo variants are normalized through `TYPO_MAP`.
   - Purpose: ensure semantic label consistency and prevent duplicated classes due to naming noise.

3. **Manual relabeling**
   - `RELABEL_MAP` contains confirmed manual corrections from visual inspection.

4. **Manual removals**
   - `REMOVE_LIST` includes videos removed after inspection where labels/quality were not acceptable.
   - Current list targets reviewed `others` samples.

5. **Class exclusions (kept configurable)**
   - `DOMAIN_EXCLUDE_TYPES = []` by default.
   - `POLICY_EXCLUDE_TYPES = []` by default.
   - `EXCLUDE_TYPES` is computed from both lists and applied only when non-empty.

### Imbalance monitoring implication
After these decisions, class distribution is recalculated from the **final train split** for audit and reporting.

### Reproducibility outputs
This notebook exports:
- cleaned train/valid CSVs,
- `decisions_manifest.json` capturing policy + applied decisions.

### Next steps
- Run parallel modeling tracks with MediaPipe and YOLOv8 using these cleaned splits.
- Compare per-class MAE and FPS across both approaches.
- Use OpenPose as an optional benchmark on a small subset if needed.

### Class-level review log
| class        | status  | note                                      |
|--------------|---------|-------------------------------------------|
| others       | Removed | OOD/noisy cases from manual review        |
| rowing_erg   | Kept    | No active low-sample exclusion in default |


## Data Leakage Checks

### Why this section exists
This section verifies that train/validation boundaries are clean and that preprocessing decisions are not introducing leakage.

### Leakage checks performed
1. **Exact split overlap by `name`**
   - Detects videos appearing in both train and validation.
2. **Exact row overlap (`type`, `name`, `count`)**
   - Detects duplicated labeled records across splits.
3. **Near-duplicate base-name overlap**
   - Detects likely related clips such as `train123.mp4` and `train123_1.mp4` crossing splits.
4. **Curation policy scope checks**
   - Confirms relabel/remove lists do not unexpectedly hit validation beyond intended policy.
5. **Summary verdict**
   - Prints PASS/WARN flags for quick audit.

### Interpretation
- Any non-zero overlap should be reviewed before modeling.
- Near-duplicate overlap is a warning signal and may require manual review.
- This section is a guardrail, not proof of zero leakage for all hidden metadata.



In [ ]:
import re

# --- helper functions ---
def _safe_series(df, col):
    return df[col].astype(str).str.strip() if col in df.columns else pd.Series(dtype=str)

def _normalize_name(s):
    s = str(s).strip()
    return s.lower()

def _base_video_name(name):
    # Remove extension, then common clip suffix patterns (e.g., _1, -2)
    n = str(name).strip().lower()
    n = re.sub(r'\.mp4$', '', n)
    n = re.sub(r'([_-]clip)?[_-]?\d+$', '', n)
    return n

# --- 1) exact overlap by video name ---
train_names = set(_safe_series(train_df, 'name').map(_normalize_name))
valid_names = set(_safe_series(valid_df, 'name').map(_normalize_name))
name_overlap = sorted(train_names & valid_names)

print('Leakage check 1/5 — exact name overlap (train vs valid):')
print(f'  overlap_count = {len(name_overlap)}')
if len(name_overlap) > 0:
    print('  sample_overlaps:', name_overlap[:20])

# --- 2) exact row overlap by key columns ---
key_cols = [c for c in ['type', 'name', 'count'] if c in train_df.columns and c in valid_df.columns]
if key_cols:
    train_keys = set(map(tuple, train_df[key_cols].astype(str).values.tolist()))
    valid_keys = set(map(tuple, valid_df[key_cols].astype(str).values.tolist()))
    row_overlap = train_keys & valid_keys
else:
    row_overlap = set()

print('Leakage check 2/5 — exact row overlap (type,name,count):')
print(f'  overlap_count = {len(row_overlap)}')

# --- 3) near-duplicate overlap using base names ---
train_base = set(_safe_series(train_df, 'name').map(_base_video_name))
valid_base = set(_safe_series(valid_df, 'name').map(_base_video_name))
base_overlap = sorted(train_base & valid_base)

print('Leakage check 3/5 — near-duplicate base-name overlap:')
print(f'  overlap_count = {len(base_overlap)}')
if len(base_overlap) > 0:
    print('  sample_base_overlaps (no human repeated visibly):', base_overlap[:20])

# --- 4) policy-scope checks for curation lists ---
# This notebook policy is train+valid scope; test is out-of-scope here.
relabel_norm_local = {str(k).strip().lower() if str(k).strip().lower().endswith('.mp4') else f"{str(k).strip().lower()}.mp4": v
                     for k, v in RELABEL_MAP.items()} if 'RELABEL_MAP' in globals() else {}
remove_set_local = {str(x).strip().lower() if str(x).strip().lower().endswith('.mp4') else f"{str(x).strip().lower()}.mp4"
                    for x in REMOVE_LIST} if 'REMOVE_LIST' in globals() else set()

valid_name_norm = set(_safe_series(valid_df, 'name').map(_normalize_name))
valid_relabel_hits = sorted([n for n in relabel_norm_local if n in valid_name_norm])
valid_remove_hits = sorted([n for n in remove_set_local if n in valid_name_norm])

print('Leakage check 4/5 — curation policy scope audit (validation hits):')
print(f'  relabel_hits_in_valid = {len(valid_relabel_hits)}')
print(f'  remove_hits_in_valid  = {len(valid_remove_hits)}')

# --- 5) verdict ---
exact_overlap_pass = len(name_overlap) == 0 and len(row_overlap) == 0
near_dup_warn = len(base_overlap) > 0

print('Leakage check 5/5 — summary verdict:')
print('  exact_split_overlap_pass =', exact_overlap_pass)
print('  near_duplicate_warning   =', near_dup_warn)

if exact_overlap_pass and not near_dup_warn:
    print('LEAKAGE STATUS: PASS (no exact/near-duplicate signals from current checks).')
elif exact_overlap_pass and near_dup_warn:
    print('LEAKAGE STATUS: WARN (no exact overlap, but near-duplicate candidates exist).')
else:
    print('LEAKAGE STATUS: FAIL (exact overlap detected; fix before modeling).')



## 5.1) Decision Impact Check (Before vs After)

**Why this section**
- Any class refactor (relabel/remove/exclude) changes the dataset distribution.
- Imbalance should be recomputed on the **final train set**, not on pre-clean counts.

**Decision(s)**
- Compare train class counts before decisions (`train_counts_after_typo`) vs final (`train_df`).
- Report train imbalance ratio before and after.

**Next step(s)**
- Use post-decision imbalance numbers for downstream modeling diagnostics.
- Re-run this section whenever decision lists or exclusions change.


In [ ]:
final_train_counts = train_df['type'].value_counts().sort_values(ascending=False)

cmp = pd.concat([
    train_counts_after_typo.rename('before_decisions'),
    final_train_counts.rename('after_decisions')
], axis=1).fillna(0).astype(int)
cmp['delta'] = cmp['after_decisions'] - cmp['before_decisions']

print('Class count impact (train):')
print(cmp.to_string())

before_ratio = train_counts_after_typo.max() / train_counts_after_typo.min()
after_ratio = final_train_counts.max() / final_train_counts.min()
print(f"\nImbalance ratio (train) before decisions: {before_ratio:.2f}x")
print(f"Imbalance ratio (train) after decisions : {after_ratio:.2f}x")



## 6) Build Clean Analysis Frame and Audit

**Why this section**
- Confirms final data integrity before modeling: split sizes, labels, and unresolved classes.

**Decision(s)**
- Build `df_tv = train + valid` for analysis/modeling context.
- Drop rows with missing `count` in train+valid.
- Verify `others`/`rowing_erg` status post-decisions.

**Next step(s)**
- Stop modeling if audit checks fail (unexpected labels, unresolved classes, null targets).


In [ ]:
df_tv = pd.concat([train_df, valid_df], ignore_index=True)
missing_count = int(df_tv['count'].isna().sum())
if missing_count > 0:
    print(f"Dropping {missing_count} rows with missing count from train+valid")

df_tv = df_tv.dropna(subset=['count']).copy()

print('\nFinal split sizes:')
print(f"  train={len(train_df)}, valid={len(valid_df)}")
print(f"  train+valid (count non-null)={len(df_tv)}")

print('\nFinal train label distribution:')
print(train_df['type'].value_counts().to_string())

print('\nRemaining `others` in train+valid:', int((df_tv['type']=='others').sum()))
print('Remaining `rowing_erg` in train+valid:', int((df_tv['type']=='rowing_erg').sum()))



## 7) Export Cleaned CSVs

**Why this section**
- Downstream stages should consume frozen, versioned cleaned artifacts rather than raw files.

**Decision(s)**
- Export cleaned train/valid CSVs and a reproducibility manifest.

**Next step(s)**
- Point MediaPipe and YOLOv8 data loading paths to `annotation_cleaned/` outputs.
- Keep experiment/version metadata aligned across both modeling tracks.


In [ ]:
import json
from datetime import datetime, timezone

train_out = os.path.join(OUTPUT_DIR, 'train_cleaned.csv')
valid_out = os.path.join(OUTPUT_DIR, 'valid_cleaned.csv')
manifest_out = os.path.join(OUTPUT_DIR, 'decisions_manifest.json')

train_df.to_csv(train_out, index=False)
valid_df.to_csv(valid_out, index=False)

manifest = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'input_paths': {
        'train': TRAIN_PATH,
        'valid': VALID_PATH,
    },
    'output_paths': {
        'train_cleaned': train_out,
        'valid_cleaned': valid_out,
    },
    'policy': {
        'scope': 'train_valid_only',
        'typo_map_applied_to': ['train', 'valid'],
        'manual_relabel_applied_to': ['train', 'valid'],
        'remove_list_applied_to': ['train', 'valid'],
        'exclude_types_applied_to': ['train', 'valid'] if EXCLUDE_TYPES else [],
        'test_policy': 'not handled in this notebook',
    },
    'decisions': {
        'domain_exclude_types': DOMAIN_EXCLUDE_TYPES,
        'policy_exclude_types': POLICY_EXCLUDE_TYPES,
        'exclude_types': EXCLUDE_TYPES,
        'relabel_map': RELABEL_MAP,
        'remove_list': REMOVE_LIST,
    },
    'changes_summary': {
        'manual_relabel_rows': affected_rows,
        'excluded_rows': excluded_rows,
        'dropped_valid_all_null_columns': dropped_valid_null_cols,
    },
    'row_counts': {
        'train_cleaned': int(len(train_df)),
        'valid_cleaned': int(len(valid_df)),
    },
}

with open(manifest_out, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=True, indent=2)

print('Saved:')
print(' ', train_out)
print(' ', valid_out)
print(' ', manifest_out)


## 8) Data Preparation Summary

### What was done
- Standardized noisy labels using `TYPO_MAP`.
- Applied manual relabel/removal decisions from inspection (`RELABEL_MAP`, `REMOVE_LIST`).
- Kept exclusion hooks configurable and empty by default (`DOMAIN_EXCLUDE_TYPES`, `POLICY_EXCLUDE_TYPES`).
- Recomputed post-decision class distribution and imbalance ratio for audit.
- Exported cleaned train/valid datasets and a reproducibility manifest.

### Final modeling scope
- This notebook is **train+valid only**.
- Test is intentionally out of scope for this stage.

### Exported artifacts
- `train_cleaned.csv`
- `valid_cleaned.csv`
- `decisions_manifest.json`

### Readiness check
- Labels normalized and curated.
- Split-boundary leakage checks added (train vs valid).
- Artifacts versioned via manifest.

### Next step
Proceed to modeling notebooks for MediaPipe and YOLOv8, then compare results with a shared evaluation template.
